## Training a Neural Network

In this exercise, you will have to implement all aspects of a neural network training script. This includes creating data loading and model training, testing logic in your code. Here are the steps you need to do to complete this exercise:

1. Finish the `train()` and `test()` functions. These should contain loops to train and test your model respectively.
2. Finish the `create_model()` function. You can use the same model from a previous exercise.
3. Create Data Transforms and Data Loaders to download, read and preprocess your data.
4. Add a cost function and optimizer. You can use the same cost functions and optimizer from the previous exercise.

**Note**: It may take 5 - 10 minutes to download the data sets. 

In case you get stuck, you can look at the solution notebook.

In [17]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [10]:
#TODO: Import packages you need

def create_model():
    #TODO: Add your model code here. You can use code from previous exercises
    input_size = 784  #28x28
    output_size = 10
    model = nn.Sequential(nn.Linear(input_size, 256), #Performs W.x + b
                          nn.ReLU(),                  #Adds Non-Linearity
                          nn.Linear(256, 128),
                          nn.ReLU(),
                          nn.Linear(128, output_size),
                          nn.LogSoftmax(dim=1))
    return model

In [11]:
def train(model, train_loader, cost, optimizer, epoch, device="cpu"):
    model.to(device)
    model.train()
    for e in range(epoch):
        running_loss = 0
        correct = 0
        total = 0
        for data, target in train_loader:
            # Move batch to CPU/GPU
            data, target = data.to(device), target.to(device)
            # Flatten images: [batch, 1, 28, 28] -> [batch, 784]
            data = data.view(data.shape[0], -1)
            # Clear gradients from previous batch
            optimizer.zero_grad()
            # Forward pass
            pred = model(data)
            # Calculate loss
            loss = cost(pred, target)
            running_loss += loss.item()
            # Backpropagation
            loss.backward()
            # Update weights
            optimizer.step()
            # Get predicted class
            pred = pred.argmax(dim=1, keepdim=True)
            # Count correct predictions
            correct += pred.eq(target.view_as(pred)).sum().item()
            total += target.size(0)
        avg_loss = running_loss / len(train_loader)
        avg_accuracy = 100 * correct / total

        print(
            f"Epoch {e+1}: "
            f"Loss {avg_loss:.4f}, "
            f"Accuracy {avg_accuracy:.2f}%"
        )

In [12]:
def test(model, test_loader, device="cpu"):
    model.to(device)
    model.eval()

    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            # Move batch to CPU/GPU
            data, target = data.to(device), target.to(device)

            # Flatten: [batch, 1, 28, 28] -> [batch, 784]
            data = data.view(data.shape[0], -1)

            # Forward pass
            output = model(data)

            # Get predicted class
            pred = output.argmax(dim=1, keepdim=True)

            # Count correct predictions
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_accuracy = 100 * correct / len(test_loader.dataset)

    print(
        f"Test set: Accuracy: {correct}/{len(test_loader.dataset)} "
        f"({test_accuracy:.2f}%)"
    )

In [13]:
#TODO: Create your Data Transforms
training_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),  # Randomly flips the image horizontally with a probability of 0.5
    transforms.ToTensor(),  # Converts the image to a PyTorch tensor
    transforms.Normalize((0.1307,), (0.3081,))
])

testing_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [14]:
#TODO: Download and create loaders for your data
trainset = datasets.MNIST("data/", download=True, train=True, transform=training_transform)
testset = datasets.MNIST("data/", download=True, train=False, transform=testing_transform)

train_loader = DataLoader(trainset, batch_size, shuffle=True)
test_loader = DataLoader(testset, batch_size, shuffle=False)

In [18]:
batch_size = 64
epochs = 10

model=create_model()

cost = nn.NLLLoss() #TODO: Add your cost function here. You can use code from previous exercises

optimizer = optim.Adam(model.parameters(), lr=1e-3) #TODO: Add your optimizer here. You can use code from previous exercises


train(model, train_loader, cost, optimizer, epochs)

Epoch 1: Loss 0.3404, Accuracy 89.40%
Epoch 2: Loss 0.1594, Accuracy 94.95%
Epoch 3: Loss 0.1212, Accuracy 96.12%
Epoch 4: Loss 0.1014, Accuracy 96.76%
Epoch 5: Loss 0.0873, Accuracy 97.20%
Epoch 6: Loss 0.0754, Accuracy 97.60%
Epoch 7: Loss 0.0665, Accuracy 97.87%
Epoch 8: Loss 0.0592, Accuracy 98.07%
Epoch 9: Loss 0.0540, Accuracy 98.22%
Epoch 10: Loss 0.0498, Accuracy 98.38%


In [19]:
test(model, test_loader)

Test set: Accuracy: 9671/10000 (96.71%)
